# **# 2) Building Model**

Purpose: Script to test data for stationarity using ADF and KPSS tests, building SARIMA and SARIMAX and conducting the initial walk-forward validation and forecasting

Languages: Python 3

IDE: Google Colab, file is a Jupiter Notebook notebook document (.ipynb)

Inputs: monthly_oil_gas_prices.csv

Outputs: evaluation_metrics.csv, forecasts_comparison.csv, multi_horizon_comparison.csv. All combines in the evaluation_results.zip

Packages: pandas, numpy, matplotlib, warnings, statsmodels, sci-kit learn, zipfile

Run: python SCRIPTS/1_Cleaning_and_EDA.ipynb (from the repo root) first. Then this script.

Order: Run Second, followed by 3_Evaluating_Model

In [72]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Connect to GitHub
! git clone https://github.com/oliviapang/DS_4002_Project_2
data = pd.read_csv('/content/DS_4002_Project_2/DATA/monthly_oil_gas_prices.csv')
display(data.head())

fatal: destination path 'DS_4002_Project_2' already exists and is not an empty directory.


,Date,Avg_Gas_Price,Avg_Oil_Price,Log_Gas_Price,Log_Oil_Price
0,1990-08-01,1.2180,26.7560,0.197210,3.286759
1,1990-09-01,1.2580,33.5075,0.229523,3.511769
2,1990-10-01,1.3354,36.1700,0.289231,3.588230
3,1990-11-01,1.3240,32.6700,0.280657,3.486457
4,1990-12-01,1.3410,27.2200,0.293416,3.303952


In [73]:
#Analysis Plan Action Item 1
#Adjusting Crude Oil to be t-1 to use for the rest of analysis
data['Log_Oil_Price'] = data['Log_Oil_Price'].shift(1)
display(data.head())

,Date,Avg_Gas_Price,Avg_Oil_Price,Log_Gas_Price,Log_Oil_Price
0,1990-08-01,1.2180,26.7560,0.197210,NaN
1,1990-09-01,1.2580,33.5075,0.229523,3.286759
2,1990-10-01,1.3354,36.1700,0.289231,3.511769
3,1990-11-01,1.3240,32.6700,0.280657,3.588230
4,1990-12-01,1.3410,27.2200,0.293416,3.486457


In [74]:
#ADF and KPSS tests on log-adjusted data
warnings.filterwarnings('ignore')

def run_stationarity_tests(series, name):
    print(f"Stationarity Tests for {name}\n")

    # ADF Test (rounded to four decimal places)
    adf_result = adfuller(series.dropna())
    print("ADF Test")
    print(f"ADF Statistic: {adf_result[0]:.4f}")
    print(f"p-value: {adf_result[1]:.4e}")
    print("ADF Critical Values:")
    for key, value in adf_result[4].items():
        print(f"  {key}: {value:.4f}")

    # Interpreting the ADF test
    if adf_result[1] < 0.05:
        print("Conclusion: reject the null. Data is stationary.\n")
    else:
        print("Conclusion: fail to reject the null. Data is non-stationary / has unit root.\n")

    # KPSS Test
    kpss_result = kpss(series.dropna(), regression='c')
    print("KPSS Test")
    print(f"KPSS Statistic: {kpss_result[0]:.4f}")
    print(f"p-value: {kpss_result[1]:.4f}")
    print("KPSS Critical Values:")
    for key, value in kpss_result[3].items():
        print(f"  {key}: {value:.4f}")

    # Interpreting the KPSS test
    if kpss_result[1] < 0.05:
        print("Conclusion: reject the null. Data is non-stationary / has unit root).\n")
    else:
        print("Conclusion: fail to reject the null. Data is stationary.\n")
    print("="*50 + "\n")

# Running the tests on Log_Gas_Price and Log_Oil_Price
run_stationarity_tests(data['Log_Gas_Price'], 'Log_Gas_Price')
run_stationarity_tests(data['Log_Oil_Price'], 'Log_Oil_Price')

Stationarity Tests for Log_Gas_Price

ADF Test
ADF Statistic: -1.0258
p-value: 7.4374e-01
ADF Critical Values:
  1%: -3.4459
  5%: -2.8684
  10%: -2.5704
Conclusion: fail to reject the null. Data is non-stationary / has unit root.

KPSS Test
KPSS Statistic: 2.7771
p-value: 0.0100
KPSS Critical Values:
  10%: 0.3470
  5%: 0.4630
  2.5%: 0.5740
  1%: 0.7390
Conclusion: reject the null. Data is non-stationary / has unit root).


Stationarity Tests for Log_Oil_Price

ADF Test
ADF Statistic: -1.6202
p-value: 4.7264e-01
ADF Critical Values:
  1%: -3.4456
  5%: -2.8683
  10%: -2.5704
Conclusion: fail to reject the null. Data is non-stationary / has unit root.

KPSS Test
KPSS Statistic: 2.3600
p-value: 0.0100
KPSS Critical Values:
  10%: 0.3470
  5%: 0.4630
  2.5%: 0.5740
  1%: 0.7390
Conclusion: reject the null. Data is non-stationary / has unit root).




In [75]:
#Both tests are in agreement that the data is non-stationary.
#Therefore we need to take the first difference of the data.
data['Log_Gas_Price_Diff'] = data['Log_Gas_Price'].diff()
data['Log_Oil_Price_Diff'] = data['Log_Oil_Price'].diff()
display(data.head())

,Date,Avg_Gas_Price,Avg_Oil_Price,Log_Gas_Price,Log_Oil_Price,Log_Gas_Price_Diff,Log_Oil_Price_Diff
0,1990-08-01,1.2180,26.7560,0.197210,NaN,NaN,NaN
1,1990-09-01,1.2580,33.5075,0.229523,3.286759,0.032313,NaN
2,1990-10-01,1.3354,36.1700,0.289231,3.511769,0.059708,0.225011
3,1990-11-01,1.3240,32.6700,0.280657,3.588230,-0.008573,0.076461
4,1990-12-01,1.3410,27.2200,0.293416,3.486457,0.012758,-0.101773


In [76]:
#Analysis Plan Action Item 2
#SARIMA and SARIMAX Modeling
#Content adapted from Google. (2026). Gemini [Large language model]. https://gemini.google.com


# Removing NaNs that came about from data modification.
modeling_data = data[['Date', 'Log_Gas_Price_Diff', 'Log_Oil_Price_Diff']].dropna().copy()
modeling_data['Date'] = pd.to_datetime(modeling_data['Date'])
modeling_data.set_index('Date', inplace=True)

# Defining variables.
y_diff = modeling_data['Log_Gas_Price_Diff']
x_diff_exog = modeling_data['Log_Oil_Price_Diff']

#Fitting the univariate SARIMA model
sarima_model = sm.tsa.statespace.SARIMAX(
    y_diff,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_results = sarima_model.fit(disp=False)
print("SARIMA Model Results")
print(sarima_results.summary())

#Fitting multivariate SARIMAX model (with difference oil price as exogenous feature)
sarimax_model = sm.tsa.statespace.SARIMAX(
    y_diff,
    exog=x_diff_exog,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarimax_results = sarimax_model.fit(disp=False)
print("SARIMAX Model Results")
print(sarimax_results.summary())

SARIMA Model Results
                                     SARIMAX Results                                      
Dep. Variable:                 Log_Gas_Price_Diff   No. Observations:                  433
Model:             SARIMAX(1, 0, 1)x(1, 0, 1, 12)   Log Likelihood                 650.601
Date:                            Sat, 24 Oct 2026   AIC                          -1291.201
Time:                                    01:02:12   BIC                          -1271.012
Sample:                                10-01-1990   HQIC                         -1283.221
                                     - 10-01-2026                                         
Covariance Type:                              opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.0677      0.084     -0.808      0.419      -0.232       0.096
ma.L1         

In [77]:
#Analysis Plan Action Item 3
#Forecasting the data and walk-forward validation
#Content adapted from Google. (2026). Gemini [Large language model]. https://gemini.google.com

clean_data = data[['Date', 'Log_Gas_Price', 'Log_Oil_Price']].copy()
clean_data['Date'] = pd.to_datetime(clean_data['Date'])
clean_data.set_index('Date', inplace=True)
clean_data = clean_data.dropna()
#Handling for incomplete month
clean_data = clean_data.loc[:'2026-09-01']

# Explicitly set index frequency to Month Begin to prevent statsmodels alignment errors
clean_data.index.freq = 'MS'

# Split: Train on everything pre-2020, Test on everything from 2020-01-01 onwards
train = clean_data.loc[:'2019-12-01']
test = clean_data.loc['2020-01-01':]

# Fitting initial baseline models on training data using log levels (d=1)
sarima_initial = sm.tsa.statespace.SARIMAX(
    train['Log_Gas_Price'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

sarimax_initial = sm.tsa.statespace.SARIMAX(
    train['Log_Gas_Price'],
    exog=train['Log_Oil_Price'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

# Walking forward validation
rows = []
res0, res1 = sarima_initial, sarimax_initial
last_gas = train["Log_Gas_Price"].iloc[-1]

for date, row in test.iterrows():
    exog_next = pd.DataFrame({"Log_Oil_Price": [row["Log_Oil_Price"]]}, index=pd.DatetimeIndex([date], freq='MS'))
    f_sarima = res0.forecast(steps=1).iloc[0]
    f_sarimax = res1.forecast(steps=1, exog=exog_next).iloc[0]

    # Converting the values back to original dollars
    rows.append({
        "date": date,
        "actual": np.exp(row["Log_Gas_Price"]),
        "sarima": np.exp(f_sarima),
        "sarimax": np.exp(f_sarimax),
        "naive": np.exp(last_gas),
    })

    new_y = pd.Series([row["Log_Gas_Price"]], index=pd.DatetimeIndex([date], freq='MS'), name="Log_Gas_Price")
    res0 = res0.append(new_y, refit=False)
    res1 = res1.append(new_y, exog=exog_next, refit=False)
    last_gas = row["Log_Gas_Price"]

# Converting to new dataframe
fc = pd.DataFrame(rows).set_index("date")
display(fc.head())

,actual,sarima,sarimax,naive
date,,,,
2020-01-01,2.54775,2.585608,2.641090,2.55500
2020-02-01,2.44200,2.560468,2.550451,2.54775
2020-03-01,2.23420,2.489020,2.435344,2.44200
2020-04-01,1.84050,2.209534,1.901790,2.23420
2020-05-01,1.86950,1.751130,1.523842,1.84050


In [78]:
import zipfile
from google.colab import files

# Download files to put into third evaluation script
files_to_download = [
    '/content/DS_4002_Project_2/DATA/forecasts_comparison.csv',
    '/content/DS_4002_Project_2/DATA/evaluation_metrics.csv',
    '/content/DS_4002_Project_2/DATA/multi_horizon_comparison.csv'
]

zip_filename = '/content/evaluation_results.zip'

# Write files into a zip archive
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_download:
        zipf.write(file, arcname=file.split('/')[-1])

print("ZIP file successfully created! Starting browser download...")
# Trigger local browser download in Google Colab
files.download(zip_filename)

ZIP file successfully created! Starting browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [79]:
from google.colab import files
#Saving differenced and cleaned data

# Create a copy of the differenced and cleaned dataset
cleaned_diff_df = data[['Date', 'Log_Gas_Price', 'Log_Oil_Price', 'Log_Gas_Price_Diff', 'Log_Oil_Price_Diff']].copy()

# Save locally to CSV
output_csv_path = '/content/cleaned_differenced_data.csv'
cleaned_diff_df.to_csv(output_csv_path, index=False)

print("Cleaned and differenced data saved locally to /content/cleaned_differenced_data.csv")

# Trigger browser download
files.download(output_csv_path)

Cleaned and differenced data saved locally to /content/cleaned_differenced_data.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>